# EuroZone GDP Nowcasting — Starting Kit

**DataCamp Challenge · CentraleSupélec / HI-Paris**

This notebook walks you through:
1. Loading and exploring the dataset
2. Understanding the task and evaluation metric
3. Training a baseline model
4. Submitting your first solution


## 1. Setup

In [ ]:
import os, sys, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Make bench_utils importable from the notebook
sys.path.insert(0, os.path.join('ingestion_program'))
from bench_utils import load_train_data, load_test_data

DATA_DIR = os.path.join('dev_phase', 'input_data')
REF_DIR  = os.path.join('dev_phase', 'reference_data')

print('Setup complete.')

## 2. Load the raw data

In [ ]:
train_features_raw = pd.read_csv(os.path.join(DATA_DIR, 'train', 'train_features.csv'))
train_labels_raw   = pd.read_csv(os.path.join(DATA_DIR, 'train', 'train_labels.csv'))

print('Train features:', train_features_raw.shape)
train_features_raw.head()

In [ ]:
print('Train labels:', train_labels_raw.shape)
train_labels_raw.head()

## 3. Exploratory Data Analysis

In [ ]:
# Merge for EDA
train = train_features_raw.merge(train_labels_raw[['country','quarter_end','GDP_growth']],
                                  on=['country','quarter_end'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(train['GDP_growth'], bins=60, edgecolor='k', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', label='Zero growth')
ax.set_xlabel('GDP growth (QoQ %)')
ax.set_ylabel('Count')
ax.set_title('Distribution of quarterly GDP growth (train set, all 9 countries)')
ax.legend()
plt.tight_layout()
plt.show()

print('Mean growth:', round(train['GDP_growth'].mean(), 3), '%')
print('Std  growth:', round(train['GDP_growth'].std(),  3), '%')
print('Min  growth:', round(train['GDP_growth'].min(),  3), '%')
print('Max  growth:', round(train['GDP_growth'].max(),  3), '%')

In [ ]:
# GDP growth over time for each country
countries = sorted(train['country'].unique())
fig, axes = plt.subplots(3, 3, figsize=(15, 9), sharex=True)
axes = axes.flatten()

for ax, c in zip(axes, countries):
    sub = train[train['country'] == c].sort_values('quarter_end')
    ax.plot(pd.to_datetime(sub['quarter_end']), sub['GDP_growth'], lw=1)
    ax.axhline(0, color='red', linestyle='--', lw=0.7)
    ax.set_title(c, fontweight='bold')
    ax.set_ylabel('GDP growth (%)')

fig.suptitle('Quarterly GDP growth by country (train: 2000–2015)', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between BCI and GDP growth
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat, title in zip(axes,
                            ['BCI_m3', 'CCI_m3', 'UNETOT_m3'],
                            ['Business Confidence (last month of quarter)',
                             'Consumer Confidence (last month)',
                             'Unemployment rate (last month)']):
    ax.scatter(train[feat], train['GDP_growth'], alpha=0.3, s=10)
    corr = train[[feat, 'GDP_growth']].corr().iloc[0, 1]
    ax.set_xlabel(feat)
    ax.set_ylabel('GDP growth')
    ax.set_title(f'{title}\n(r = {corr:.2f})')

plt.tight_layout()
plt.show()

## 4. The Task and Metric

This is a **regression** task. Given monthly macro indicators for a quarter, predict the quarter-on-quarter **GDP growth rate** (in %).

The primary metric is **RMSE** (Root Mean Squared Error):

$$
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2}
$$

**Lower is better.** A naive baseline (predicting the mean) gives RMSE ≈ std(y_train) ≈ 1.3.

## 5. Baseline Model

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Load pre-processed data via bench_utils
X_train, y_train = load_train_data(os.path.join(DATA_DIR, 'train'))
X_test           = load_test_data(DATA_DIR, split='test')
y_test_true      = pd.read_csv(os.path.join(REF_DIR, 'test_labels.csv'))['GDP_growth'].values

model = Pipeline([
    ('scaler', StandardScaler()),
    ('reg', GradientBoostingRegressor(n_estimators=200, max_depth=4,
                                       learning_rate=0.05, subsample=0.8, random_state=42)),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
rmse   = math.sqrt(mean_squared_error(y_test_true, y_pred))
print(f'Test RMSE: {rmse:.4f}')

In [ ]:
# Predicted vs actual
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(y_test_true, y_pred, alpha=0.5, s=15)
lims = [min(y_test_true.min(), y_pred.min()) - 0.5,
        max(y_test_true.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, 'r--', lw=1, label='Perfect prediction')
ax.set_xlabel('Actual GDP growth (%)')
ax.set_ylabel('Predicted GDP growth (%)')
ax.set_title(f'Baseline GBR — Test set (RMSE={rmse:.3f})')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importances = model.named_steps['reg'].feature_importances_
feat_names  = X_train.columns.tolist()

imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1])
ax.set_xlabel('Feature importance')
ax.set_title('Top 15 features (Gradient Boosting)')
plt.tight_layout()
plt.show()

## 7. Prepare Your Submission

Write your `submission.py` with a `get_model()` function, then run the local pipeline:

```bash
python ingestion_program/ingestion.py \
    --data-dir    dev_phase/input_data \
    --output-dir  ingestion_res \
    --submission-dir solution

python scoring_program/scoring.py \
    --reference-dir dev_phase/reference_data \
    --prediction-dir ingestion_res \
    --output-dir scoring_res
```

Then zip and upload: `zip my_submission.zip submission.py`

**Ideas to try:**
- XGBoost / LightGBM
- Feature engineering: momentum (`m3 - m1`), cross-country averages
- Country one-hot encoding instead of label encoding
- Hyperparameter search with cross-validation
